In [4]:
from bs4 import BeautifulSoup

html = """
<html>
  <body>
    <a href="#item_1">Item 1. Business</a>
    <h2 id="item_1">Item 1. Business</h2>
    <p>Microsoft develops software and services.</p>
    <p>Its customers include businesses and individuals.</p>
  </body>
</html>
"""

soup = BeautifulSoup(html, "html.parser")
link = soup.find("a")
print(link)
print(link.get("href"))
heading = soup.find(id="item_1")
print(heading.get_text(" ", strip=True))
paragraphs = soup.find_all("p")

for paragraph in paragraphs:
    print(paragraph.get_text(" ", strip=True))

<a href="#item_1">Item 1. Business</a>
#item_1
Item 1. Business
Microsoft develops software and services.
Its customers include businesses and individuals.


In [5]:
from pathlib import Path
from bs4 import BeautifulSoup

path = Path(r"C:\Users\admin\PycharmProjects\Deeplearning_Study\rag_agent_system\data\sec\raw\microsoft\msft-2025-10k.html")
soup = BeautifulSoup(path.read_bytes(), "html.parser")

for section_id in ("item_1_business", "item_1a_risk_factors"):
    heading = soup.find(id=section_id)

    if heading is None:
        print("未找到：", section_id)
        continue

    print("标签：", heading.name)
    print("ID：", heading.get("id"))
    print("文字：", heading.get_text(strip=True))
    print()

标签： p
ID： item_1_business
文字： ITEM 1. BUSINESS

标签： p
ID： item_1a_risk_factors
文字： ITEM 1A. RISK FACTORS



In [6]:
def extract_paragraphs(
    soup: BeautifulSoup,
    start_id: str,
    end_id: str,
) -> list[str]:
    result = []
    found_end = False
    start_heading = soup.find(id=start_id)
    end_heading = soup.find(id=end_id)
    if start_heading is None or end_heading is None:
        raise ValueError(f"Missing section boundary: {start_id} or {end_id}")
    for paragraph in start_heading.find_all_next("p"):
        if paragraph is end_heading:
            found_end=True
            break
        elif paragraph.find_parent("div", class_="main-content-container") is None:
            continue
        else:
            text = " ".join(paragraph.get_text().split())
            if text:
                result.append(text)
    if not found_end:
        raise ValueError(f"End section {end_id} was not reached after {start_id}")
    return result

In [7]:
sections = {
    "Item 1A": (
        "item_1a_risk_factors",
        "item_1b_unresolved_staff_comments",
    ),
    "Item 7": (
        "item_7_managements_discussion_analysis_f",
        "item_7a_quantitative_qualitative_disclos",
    ),
}

for name, (start_id, end_id) in sections.items():
    paragraphs = extract_paragraphs(soup, start_id, end_id)
    print(name, "段落数：", len(paragraphs))
    print("前两段：", paragraphs[:2])
    print("最后两段：", paragraphs[-2:])

Item 1A 段落数： 75
前两段： ['Our operations and financial results are subject to various risks and uncertainties, including those described below, that could adversely affect our business, operations, financial condition, results of operations, liquidity, and the trading price of our common stock.', 'STRATEGIC AND COMPETITIVE RISKS']
最后两段： ['In addition, our international growth strategy includes certain markets, the developing nature of which presents several risks, including deterioration of social, political, labor, or economic conditions in a country or region, and difficulties in staffing and managing foreign operations. Emerging nationalist and protectionist trends and concerns about human rights, the environment, and political expression in specific countries may significantly alter the trade and commercial environments. Changes to trade policy or agreements as a result of populism, protectionism, or economic nationalism may result in higher tariffs, local sourcing initiatives, and no

In [8]:
from urllib.parse import urlsplit

href = "https://example.com/report.htm#item_1_business"
print(urlsplit(href).fragment)

item_1_business


In [ ]:
def find_section_anchor(
    soup: BeautifulSoup,
    title: str,
) -> str:
    links = soup.find_all("a",href=True)
    expected_title = " ".join(title.split())
    anchors: set[str] = set()
    for link in links:
        text = " ".join(link.get_text().split())
        if text != expected_title:
            continue
        else:
            anchor=urlsplit(link.get("href")).fragment
            if not anchor:
                raise ValueError(f"Missing URL fragment for title: {title}")
            if soup.find(id=anchor) is None:
                raise ValueError(f"Missing anchor for {link}")
            anchors.add(anchor)
    if len(anchors) != 1:
        raise ValueError(f"Found {len(anchors)} anchors for {title}")
    return anchors.pop()

In [ ]:
from bs4 import BeautifulSoup, Tag

def collect_section_elements(
    soup: BeautifulSoup,
    start_id: str,
    end_id: str,
) -> list[Tag]:
    result = []
    found_end = False
    start_heading = soup.find(id=start_id)
    end_heading = soup.find(id=end_id)
    if start_heading is None or end_heading is None:
        raise ValueError(f"Missing section boundary: {start_id} or {end_id}")
    for paragraph in start_heading.find_all_next():
        if paragraph is end_heading:
            found_end=True
            break
        result.append(paragraph)
    if not found_end:
        raise ValueError(f"End section {end_id} was not reached after {start_id}")
    return result

In [ ]:
def select_content_blocks(elements: list[Tag]) -> list[Tag]:
    result = []
    for element in elements:
        if element.find_parent("table") is not None:
            continue
        if element.name == "table":
            result.append(element)
            continue
        if (
            element.name in {"p", "div", "li"}
            and element.find(["p", "div", "table", "ul", "ol", "li"]) is None
        ):
            result.append(element)
    return result

In [ ]:
def is_noise_block(block: Tag, ticker: str) -> bool:
    text = " ".join(block.get_text(" ", strip=True).split())
    has_toc_link = any(
    " ".join(link.get_text().split()) == "Table of Contents"
    for link in block.find_all("a", href=True)
)
    if not text:
        return True
    if ticker == "MSFT":
        if block.find_parent("div", class_="main-content-container") is None:
            return True
    else:
        for ancestor in block.parents:
            style = ancestor.get("style", "")
            declarations = style.lower().replace(" ", "").split(";")
            if "position:absolute" in declarations and "bottom:0" in declarations:
                return True
    if ticker == "NVDA":
        if text == "Table of Contents" and has_toc_link:
            return True
    if ticker == "GOOG":
        if text == "Table of Contents Alphabet Inc." and has_toc_link:
            return True
    return False